In [ ]:
# ================== CELL BREAK ==================
# Cell 1: ⚙️ Global Settings and Environment Preparation

import os
import time
import random
import gc
from pathlib import Path
from glob import glob
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns


# --- Core Configuration ---
class Config:
    USE_GOOGLE_DRIVE = True
    DRIVE_PATH = '/content/drive/MyDrive/Colab/VisionTransformer'
    LEARNING_RATE = 5e-4
    SEED = 16

cfg = Config()

# --- Environment Setup ---
if cfg.USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = str(Path(cfg.DRIVE_PATH) / 'data')
    OUTPUT_ROOT = str(Path(cfg.DRIVE_PATH) / 'outputs')
else:
    DATA_ROOT = './data'
    OUTPUT_ROOT = './outputs'

Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[INFO] Global seed set to {seed}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")


In [ ]:
# ================== CELL BREAK ==================
# Cell 2: Model architecture

class AttnSaveEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        self.attn_map = None
    def forward(self, src):
        attn_output, attn_weights = self.self_attn(src, src, src, need_weights=True)
        self.attn_map = attn_weights.detach().cpu()
        src = src + self.dropout1(attn_output)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

class PatchEmbed(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, embed_dim):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x); x = x.flatten(2); x = x.transpose(1, 2); return x

class SimpleTransformerClassifier(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, n_heads, n_layers, hidden_dim, n_classes):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, hidden_dim)
        self.n_patches = self.patch_embed.n_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, hidden_dim))
        self.encoder_layers = nn.ModuleList([AttnSaveEncoderLayer(hidden_dim, n_heads) for _ in range(n_layers)])
        self.classifier = nn.Linear(hidden_dim * self.n_patches, n_classes)
        self.latest_attn_maps = None
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        attn_maps = []
        for layer in self.encoder_layers:
            x = layer(x); attn_maps.append(layer.attn_map)
        out = x.contiguous().view(x.size(0), -1)
        self.latest_attn_maps = attn_maps
        return self.classifier(out)




In [ ]:
# ================== CELL BREAK ==================
# Cell 3: Core Functions

def get_flat_grad(model):
    """ Get flattened gradients of all model parameters """
    grads = []
    for p in model.parameters():
        if p.grad is not None:
            grads.append(p.grad.detach().flatten().cpu())
    return torch.cat(grads) if grads else torch.tensor([], dtype=torch.float32)

def compute_grad_entropy(grad_tensor: torch.Tensor) -> float:
    """ Calculate normalized gradient entropy """
    g = torch.abs(grad_tensor)
    g_sum = g.sum()
    if g_sum == 0 or len(g) <= 1:
        return 0.0
    probs = g / g_sum
    log_probs = torch.log2(probs + 1e-10)
    entropy = -(probs * log_probs).sum()
    entropy_max = torch.log2(torch.tensor(len(g), dtype=torch.float32))
    return float(entropy / entropy_max)

def compute_attention_entropy(attn_maps, bins=30):
    """ Calculate entropy of multi-layer attention maps """
    ent = []
    if attn_maps is None: return ent
    for amap in attn_maps:
        if amap is not None:
            arr = amap.flatten().numpy()
            hist, _ = np.histogram(arr, bins=bins, range=(0, 1))
            if hist.sum() > 0:
                probs = hist / hist.sum()
                ent.append(float(-(probs[probs > 0] * np.log(probs[probs > 0])).sum()))
            else: ent.append(0.0)
    return ent

@torch.no_grad()
def evaluate_avg_loss_and_acc(model, data_loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for images, labels in data_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(data_loader), total_correct / total if total > 0 else 0.0


In [ ]:
# ================== CELL BREAK ==================
# Cell 4: Data Loading and Preprocessing (Unchanged)

def get_dataloaders_and_config(dataset_name: str, batch_size: int, data_root: str):
    if dataset_name == 'mnist':
        config = {'img_size': 28, 'patch_size': 4, 'in_chans': 1, 'n_classes': 10}
        transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
        train_data = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
        test_data = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
    elif dataset_name == 'cifar10':
        config = {'img_size': 32, 'patch_size': 4, 'in_chans': 3, 'n_classes': 10}
        train_transform = transforms.Compose([transforms.RandomCrop(32,padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
        test_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
        train_data = datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_transform)
        test_data = datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_transform)
    elif dataset_name == 'cifar100':
        config = {'img_size': 32, 'patch_size': 4, 'in_chans': 3, 'n_classes': 100}
        train_transform = transforms.Compose([transforms.RandomCrop(32,padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761))])
        test_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761))])
        train_data = datasets.CIFAR100(root=data_root, train=True, download=True, transform=train_transform)
        test_data = datasets.CIFAR100(root=data_root, train=False, download=True, transform=test_transform)
    else: raise ValueError("Unknown dataset")
    g = torch.Generator().manual_seed(42);
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2, generator=g)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
    return train_loader, test_loader, config


In [ ]:
# ================== CELL BREAK ==================
# Cell 5: Main Training Function (Entropy calculation and model saving fully restored)

def run_for_once(model, optimizer, criterion, train_loader, test_loader, device, n_layers, epochs, log_csv_path, save_model_path, scheduler=None):
    model.to(device)
    logs_list = []
    sample_imgs, _ = next(iter(train_loader))
    sample_imgs = sample_imgs.to(device)
    best_val_acc = -1.0
    best_model_state = None

    for ep in (pbar_epoch := tqdm(range(epochs), desc="Epochs")):
        ep_start_time = time.time()
        model.train()
        running_loss, train_correct, train_total, flat_grad = 0.0, 0, 0, None

        for bi, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            if (bi + 1) == len(train_loader):
                flat_grad = get_flat_grad(model)

            running_loss += loss.item()
            train_correct += (outputs.argmax(dim=1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = train_correct / train_total
        val_loss, val_acc = evaluate_avg_loss_and_acc(model, test_loader, device)

        if scheduler: scheduler.step()

        # Calculate Entropy
        grad_entropy = compute_grad_entropy(flat_grad)
        with torch.no_grad():
            _ = model(sample_imgs)
            attn_entropy_list = compute_attention_entropy(model.latest_attn_maps)
        attn_entropy_list.extend([0.0] * (n_layers - len(attn_entropy_list)))

        # Log entropy metrics
        log_row = {
            "epoch": ep + 1, "train_loss": train_loss, "val_loss": val_loss,
            "train_acc": train_acc, "val_acc": val_acc,
            "learning_rate": optimizer.param_groups[0]['lr'],
            "epoch_time_sec": time.time() - ep_start_time,
            "grad_entropy": grad_entropy,
            "attn_entropy_avg": np.mean(attn_entropy_list),
            **{f"attn_entropy_layer_{i+1}": v for i, v in enumerate(attn_entropy_list)}
        }
        logs_list.append(log_row)

        pbar_epoch.set_postfix_str(f"trA {train_acc:.3f} vaA {val_acc:.3f} GradH {grad_entropy:.3f}")

        # Track and save the best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()

    logs_df = pd.DataFrame(logs_list)
    logs_df['cum_time_sec'] = logs_df['epoch_time_sec'].cumsum()
    logs_df.to_csv(log_csv_path, index=False)

    # After training, save best model weights to .pth file
    if best_model_state is not None:
        torch.save({
            "model_state": best_model_state,
            "best_val_acc": best_val_acc,
            "config": {"n_layers": n_layers, "epochs": epochs}
        }, save_model_path)
        print(f"✅ Best model saved to {save_model_path} (val_acc: {best_val_acc:.4f})")
    else:
        print("⚠️ No best model state found to save.")


In [ ]:
# ================== CELL BREAK ==================
# Cell 6: Experiment Preparation Module
# This cell defines the configuration and helper functions required to run all experiment batches.
# It only needs to be run once before starting all experiments.

ALL_OPTIMIZER_CONFIGS = {
    "Adam_noWD": {"optimizer": torch.optim.Adam, "wd": 0.0},
    "Adam_WD":   {"optimizer": torch.optim.Adam, "wd": 1e-4},
    "AdamW_WD":  {"optimizer": torch.optim.AdamW, "wd": 1e-4},
}

# --- Select the optimizers to run for this session ---
#
# Scenario 1: Run two Adam optimizers first
OPTIMIZERS_TO_RUN = ["Adam_noWD", "Adam_WD"]
#
# Scenario 2: Supplement with AdamW later
# OPTIMIZERS_TO_RUN = ["AdamW_WD"]
#
# Scenario 3: Run all three at once (if resources allow)
# OPTIMIZERS_TO_RUN = ["Adam_noWD", "Adam_WD", "AdamW_WD"]

print(f"This run will execute experiments for the following optimizers: {OPTIMIZERS_TO_RUN}")


# --- Other experiment configurations (unchanged) ---
MODEL_CONFIGS = { "M_Light":  {"n_layers": 4, "hidden_dim": 64},
                 "M_Base":   {"n_layers": 8, "hidden_dim": 128},
                  "M_Deep":   {"n_layers": 12, "hidden_dim": 128},
                  "M_Wide":   {"n_layers": 8, "hidden_dim": 256},
                  # "M_ExtraWide": {"n_layers": 8, "hidden_dim": 384},
}

SEED_LIST = [16, 42, 123, 1234, 2025] # e.g., [16, 42, 123, 2025]

EPOCHS_PER_DATASET = {
    "mnist": 30,
    "cifar10": 60,
    "cifar100": 80,
}

# Ensure criterion variable exists
try:
    criterion
except NameError:
    criterion = nn.CrossEntropyLoss()
    print("Criterion initialized.")


In [ ]:
# --- Helper function to execute a single dataset batch ---
def execute_dataset_batch(dataset_to_run):
    start_time = time.time()
    print(f"\nStarting experiments for dataset: '{dataset_to_run}' with optimizers: {OPTIMIZERS_TO_RUN}...")

    for optim_name in OPTIMIZERS_TO_RUN:
        optim_config = ALL_OPTIMIZER_CONFIGS[optim_name]
        for model_name, model_config in MODEL_CONFIGS.items():
            for seed in SEED_LIST:

                # --- [Key Correction] Dynamically construct and check filename based on optimizer type ---
                experiment_name = ""
                # [Corrected] The condition here is now "Adam_noWD"
                if optim_name == "Adam_noWD":
                    # For the default Adam (no WD), use the original filename format
                    experiment_name = f"D_{dataset_to_run}_M_{model_name}_S_{seed}"
                else:
                    # For all other optimizers, add an "_O_" tag for distinction
                    experiment_name = f"D_{dataset_to_run}_M_{model_name}_O_{optim_name}_S_{seed}"

                log_path = Path(OUTPUT_ROOT) / "grid_search" / f"{experiment_name}_logs.csv"
                if log_path.exists():
                    print(f"✅ SKIPPING: Result for {log_path.stem} already exists.")
                    continue

                print("\n" + "="*50 + f"\n▶️ STARTING EXPERIMENT: {experiment_name}\n" + "="*50)

                set_seed(seed)
                train_loader, test_loader, data_config = get_dataloaders_and_config(dataset_to_run, 128, DATA_ROOT)
                model = SimpleTransformerClassifier(img_size=data_config['img_size'], patch_size=data_config['patch_size'], in_chans=data_config['in_chans'], n_heads=8, n_layers=model_config['n_layers'], hidden_dim=model_config['hidden_dim'], n_classes=data_config['n_classes'])

                optimizer_class = optim_config["optimizer"]
                optimizer = optimizer_class(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=optim_config["wd"])

                ckpt_path = Path(OUTPUT_ROOT) / "grid_search" / f"{experiment_name}_best.pth"
                log_path.parent.mkdir(parents=True, exist_ok=True)

                run_for_once(model, optimizer, criterion, train_loader, test_loader, device, model_config['n_layers'], EPOCHS_PER_DATASET[dataset_to_run], log_path, ckpt_path, scheduler=None)

                del model, optimizer, train_loader, test_loader
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

    total_duration_mins = (time.time() - start_time) / 60
    print("\n" + "*"*50 + f"\n🎉 BATCH COMPLETE for dataset: '{dataset_to_run}'\n   Total duration (new runs): {total_duration_mins:.2f} minutes.\n" + "*"*50)

print("【Final Corrected Version V3】Experiment preparation module loaded.")


In [ ]:
# ================== CELL 6.1: Run MNIST ==================
execute_dataset_batch("mnist")


In [ ]:
# ================== CELL 6.2: Run CIFAR-10 ==================
execute_dataset_batch("cifar10")


In [ ]:
# ================== CELL 6.3: Run CIFAR-100 ==================
execute_dataset_batch("cifar100")


In [ ]:
# ================== CELL BREAK ==================
# Cell 7: 📊 结果汇总与可视化

print("Aggregating results from all log files...")
log_dir = Path(OUTPUT_ROOT) / "grid_search"
all_log_files = glob(str(log_dir / "*.csv"))
df_list = []

if not all_log_files:
    print("No log files found. Please run some experiments first in Cell 6.")
else:
    for f in all_log_files:
        parts = Path(f).stem.split('_')
        summary = {}

        # --- START FIX: Corrected parsing logic for filename ---
        # Example filename formats based on kernel state's `parts` variable:
        # 1. D_mnist_M_M_Light_S_16_logs.csv (for Adam_noWD)
        #    parts: ['D', 'mnist', 'M', 'M', 'Light', 'S', '16', 'logs']
        # 2. D_cifar100_M_M_Wide_O_Adam_WD_S_2025_logs.csv (for other optimizers like Adam_WD)
        #    parts: ['D', 'cifar100', 'M', 'M', 'Wide', 'O', 'Adam', 'WD', 'S', '2025', 'logs']

        summary["dataset"] = parts[1]

        m_idx = parts.index('M', 2) # Find 'M' marker, starting search after dataset

        try:
            o_idx = parts.index('O', m_idx + 1) # Find 'O' marker if optimizer is present

            # Model type is between 'M' and 'O'
            model_type_parts = parts[m_idx + 1 : o_idx]
            summary["model_type"] = '_'.join(model_type_parts)

            # Optimizer is between 'O' and 'S'
            s_idx = parts.index('S', o_idx + 1) # Find 'S' marker after 'O'
            optimizer_name_parts = parts[o_idx + 1 : s_idx]
            summary["optimizer"] = '_'.join(optimizer_name_parts)

            # Seed is after 'S'
            summary["seed"] = int(parts[s_idx + 1])

        except ValueError: # 'O' not found, implies Adam_noWD format
            summary["optimizer"] = "Adam_noWD"

            # Model type is between 'M' and 'S'
            s_idx = parts.index('S', m_idx + 1) # Find 'S' marker after 'M'
            model_type_parts = parts[m_idx + 1 : s_idx]
            summary["model_type"] = '_'.join(model_type_parts)

            # Seed is after 'S'
            summary["seed"] = int(parts[s_idx + 1])
        # --- END FIX: Corrected parsing logic for filename ---
        df = pd.read_csv(f)
        best_val_acc_row = df.loc[df['val_acc'].idxmax()]
        summary.update({
            "best_val_acc": best_val_acc_row['val_acc'],
            "best_epoch": best_val_acc_row['epoch'],
            "grad_entropy_at_best": best_val_acc_row.get('grad_entropy', np.nan),
            "attn_entropy_at_best": best_val_acc_row.get('attn_entropy_avg', np.nan),
            "total_time_min": df['cum_time_sec'].iloc[-1] / 60
        })
        df_list.append(summary)
    results_df = pd.DataFrame(df_list)

    summary_table = results_df.groupby(['dataset', 'model_type'])['best_val_acc'].agg(['mean', 'std']).reset_index()
    summary_table['mean'] *= 100
    summary_table['std'] *= 100

    print("\n\n--- Performance Summary (Mean ± Std %) ---")
    print(summary_table.to_string())

    print("\n\n--- Generating Plot ---")
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    sns.pointplot(data=summary_table, x='dataset', y='mean', hue='model_type', order=['mnist', 'cifar10', 'cifar100'], hue_order=['M_Light', 'M_Base', 'M_Deep', 'M_Wide'], ax=ax, dodge=True, errorbar='sd')
    ax.set_title('Model Performance Across Datasets (Mean Accuracy with Std Dev over Seeds)', fontsize=16)
    ax.set_xlabel('Dataset (Complexity Increases -->)', fontsize=12)
    ax.set_ylabel('Mean Best Validation Accuracy (%)', fontsize=12)
    ax.yaxis.grid(True)
    ax.legend(title='Model Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
if not all_log_files:
    print("No log files found. Please check the path and run some experiments first.")
else:

    for f in tqdm(all_log_files, desc="Analyzing Logs"):
        try:
            parts = Path(f).stem.replace('_logs','').split('_')
            summary = {}

            # Find 'M' marker for model type (dataset is parts[1])
            idx_M = parts.index('M')

            # --- Corrected parsing logic for summary dictionary ---
            summary['dataset'] = parts[1]

            # Try to find 'O' tag, indicating an optimizer other than Adam_noWD
            try:
                idx_O = parts.index('O')
                # If 'O' is found, model type is from idx_M to idx_O
                summary['model_type'] = '_'.join(parts[idx_M : idx_O])

                # Optimizer is from idx_O + 1 until 'S'
                idx_S = parts.index('S', idx_O + 1) # Find 'S' after 'O'
                summary['optimizer'] = '_'.join(parts[idx_O + 1 : idx_S])
                summary['seed'] = int(parts[idx_S + 1])

            except ValueError: # 'O' not found, implies Adam_noWD format
                summary['optimizer'] = 'Adam_noWD'

                # If 'O' is not found, model type is from idx_M to 'S'
                idx_S = parts.index('S', idx_M + 1) # Find 'S' after 'M'
                summary['model_type'] = '_'.join(parts[idx_M : idx_S])
                summary['seed'] = int(parts[idx_S + 1])
            # --- End of corrected parsing logic ---

            df = pd.read_csv(f)
            if not df.empty and 'val_acc' in df.columns:
                best_val_acc_row = df.loc[df['val_acc'].idxmax()]
                summary.update({
                    "best_val_acc": best_val_acc_row['val_acc'],
                    "best_epoch": best_val_acc_row['epoch'],
                    "grad_entropy_at_best": best_val_acc_row.get('grad_entropy', np.nan),
                    "attn_entropy_at_best": best_val_acc_row.get('attn_entropy_avg', np.nan),
                    "total_time_min": df['cum_time_sec'].iloc[-1] / 60
                })
                df_list.append(summary)
            else:
                print(f"WARNING: Log file {Path(f).name} is empty or missing 'val_acc'. Skipped.")

        except (ValueError, IndexError, pd.errors.EmptyDataError) as e:
            print(f"ERROR: Cannot process {Path(f).name}: {e}. Skipped.")
            continue # Continue to the next file if an error occurs

    if not df_list:
        print("Log files were found, but none could be processed successfully.")
    else:
        results_df = pd.DataFrame(df_list)


        summary_table = results_df.groupby(['dataset', 'model_type', 'optimizer'])['best_val_acc'].agg(['mean', 'std']).reset_index()
        summary_table['mean'] *= 100
        summary_table['std'] = (summary_table['std'] * 100).fillna(0)

        print("\n\n--- Performance Summary (Mean ± Std %) ---")
        print(summary_table.sort_values(by=['dataset', 'optimizer', 'model_type']).to_string())

        print("\n\n--- Generating Plot ---")
        plt.style.use('seaborn-v0_8-whitegrid')

        g = sns.catplot(
            data=summary_table,
            x='model_type',
            y='mean',
            hue='optimizer',
            col='dataset',
            kind='bar',
            order=['M_M_Light', 'M_M_Base', 'M_M_Deep', 'M_M_Wide'],
            col_order=['mnist', 'cifar10', 'cifar100'],
            height=5, aspect=1.2,
            legend_out=True
        )

        g.fig.suptitle('Model Performance Comparison Across Datasets and Optimizers', y=1.03, fontsize=16)
        g.set_axis_labels("Model Architecture", "Mean Best Validation Accuracy (%)")
        g.set_titles("Dataset: {col_name}")
        g.despine(left=True)

        try:
            g.move_legend(loc="upper right", bbox_to_anchor=(.9, .85))
        except AttributeError:
            g.legend.set_bbox_to_anchor((0.9, 0.85))

        plt.show()
